In [ ]:
from pathlib import Path
from typing import List

import torch.optim as opt
from classes import ModelConfig
from classes.model_config_helper import parameterized_config
from classes.workload_runs import WorkloadRuns
from dotenv import load_dotenv
from ldb_schema.datasets import dataset_list_dict
from ldb_schema.paths import LocalPaths
from main import init_model
from torch.optim import Optimizer
from training.dataset.dataset_creation import read_workload_runs
from training.training.checkpoint import load_checkpoint
from training.training.metrics import Metric, init_metrics
from training.training.utils import find_early_stopping_metric

load_dotenv()


# Initialize models and data loaders
def get_model(
    model_config: ModelConfig,
    statistics_file: str,
    model_dir: Path,
    sample_wl_file: Path,
    manual_model_name: str = None,
    manual_model_dir: str = None,
):
    print(
        f"Initializing model {model_config.name.NAME} with seed {model_config.seed} and load datasets",
        flush=True,
    )

    wl_runs = WorkloadRuns(train_workload_runs=None, test_workload_runs=None)
    print(wl_runs)
    model, _, _, _, feature_stats = init_model(
        config=model_config,
        statistics_file=statistics_file,
        column_statistics=None,
        workload_runs=wl_runs,
        word_embeddings=None,
    )

    # Init optimizer
    optimizer: Optimizer = opt.__dict__[model_config.optimizer_class_name](
        model.parameters(), **model_config.optimizer_kwargs
    )

    # Initialize metrics
    metrics: List[Metric] = init_metrics(model_config.loss_class_name)

    print(
        f"Try to load model with name {model_config.name.NAME} and seed {model_config.seed} from {model_dir}",
        flush=True,
    )
    (
        csv_stats,
        epochs_wo_improvement,
        epoch,
        model,
        optimizer,
        metrics,
        finished,
        found,
        rank_found,
    ) = load_checkpoint(
        model=model,
        target_path=model_dir,
        config=model_config,
        optimizer=optimizer,
        metrics=metrics,
        filetype=".pt",
        manual_model_name=manual_model_name,
        manual_model_dir=manual_model_dir,
    )

    assert found
    assert finished

    # load best checkpoint (instead of last)
    print(
        f"Load best checkpoint for model {model_config.name.NAME} and seed {model_config.seed}",
        flush=True,
    )
    early_stop_m = find_early_stopping_metric(metrics)
    model.load_state_dict(early_stop_m.best_model)

    # load database statistics
    _, database_statistics = read_workload_runs(
        [sample_wl_file],
        limit_plans_per_file=None,
        limit_plans_overall=None,
        merge_nlj_and_inlj_to_same_op=model_config.merge_nlj_and_inlj_to_same_op,
        remove_join_cond_from_plan=model_config.remove_join_cond_from_plan,
    )

    return model, feature_stats, database_statistics


target_dataset = [db for db in dataset_list_dict["zs"] if db.db_name == "imdb"][0]

config = parameterized_config(
    "zeroshot", wandb_group="baselines", include_pg_opt_workloads=True, seed=1
)
model_dir = config.get_model_dir(LocalPaths(), target_dataset).as_posix()
statistics_path = config.get_statistics(
    source_path=LocalPaths(), dataset=target_dataset, format=config.input_format
).as_posix()

manual_model_dir = f"{LocalPaths().Data.models.as_posix()}/pm-zeroshot-include_hint_idx_workloads/imdb/"
manual_model_name = "pm-zeroshot-include_hint_idx_workloads_1"

sample_wl_file = Path(
    f"{LocalPaths().Data.parsed_plans.as_posix()}/imdb_scaled1/workload_200k_s1_reduced.json"
)

model, feature_stats, database_statistics = get_model(
    model_config=config,
    statistics_file=statistics_path,
    model_dir=model_dir,
    manual_model_name=manual_model_name,
    manual_model_dir=manual_model_dir,
    sample_wl_file=sample_wl_file,
)

In [ ]:
plan1 = {
    "plan_parameters": {
        "act_card": 1,
        "act_time": 1159.646,
        "est_card": 1,
        "est_width": 64,
        "pg_cost": 461362.71,
        "op_name": "Simple Aggregate",
        "output_columns": [
            {
                "aggregation": "MIN",
                "columns": [
                    {
                        "col_id": 43,
                        "child_ref": None,
                        "table_name": "n",
                        "col_name": "name",
                    }
                ],
                "udf_name": None,
                "udf_output": "False",
                "child_ref": False,
            },
            {
                "aggregation": "MIN",
                "columns": [
                    {
                        "col_id": 43,
                        "child_ref": None,
                        "table_name": "n",
                        "col_name": "name",
                    }
                ],
                "udf_name": None,
                "udf_output": "False",
                "child_ref": False,
            },
        ],
        "act_children_card": 1918,
        "est_children_card": 1,
        "above_udf_filter": False,
        "is_udf_filter": False,
    },
    "children": [
        {
            "plan_parameters": {
                "act_card": 1918,
                "act_time": 1158.691,
                "est_card": 1,
                "est_width": 15,
                "pg_cost": 461362.69,
                "op_name": "Hash Join",
                "join": {
                    "table_name1": "mk",
                    "column_name1": "keyword_id",
                    "col_id1": 32,
                    "operator": "=",
                    "table_name2": "k",
                    "column_name2": "id",
                    "col_id2": 26,
                    "is_index_cond": False,
                },
                "output_columns": [
                    {
                        "aggregation": "None",
                        "columns": [
                            {
                                "col_id": 43,
                                "child_ref": None,
                                "table_name": "n",
                                "col_name": "name",
                            }
                        ],
                        "udf_name": None,
                        "udf_output": "False",
                        "child_ref": False,
                    }
                ],
                "act_children_card": 966195,
                "est_children_card": 43015,
                "above_udf_filter": False,
                "is_udf_filter": False,
            },
            "children": [
                {
                    "plan_parameters": {
                        "act_card": 966195,
                        "act_time": 1062.627,
                        "est_card": 43015,
                        "est_width": 19,
                        "pg_cost": 461241.33,
                        "op_name": "Hash Join",
                        "join": {
                            "table_name1": "mc",
                            "column_name1": "company_id",
                            "col_id1": 3,
                            "operator": "=",
                            "table_name2": "cn",
                            "column_name2": "id",
                            "col_id2": 14,
                            "is_index_cond": False,
                        },
                        "output_columns": [
                            {
                                "aggregation": "None",
                                "columns": [
                                    {
                                        "col_id": 32,
                                        "child_ref": None,
                                        "table_name": "mk",
                                        "col_name": "keyword_id",
                                    }
                                ],
                                "udf_name": None,
                                "udf_output": "False",
                                "child_ref": False,
                            },
                            {
                                "aggregation": "None",
                                "columns": [
                                    {
                                        "col_id": 43,
                                        "child_ref": None,
                                        "table_name": "n",
                                        "col_name": "name",
                                    }
                                ],
                                "udf_name": None,
                                "udf_output": "False",
                                "child_ref": False,
                            },
                        ],
                        "act_children_card": 227052926415,
                        "est_children_card": 10108395955,
                        "above_udf_filter": False,
                        "is_udf_filter": False,
                    },
                    "children": [
                        {
                            "plan_parameters": {
                                "act_card": 966195,
                                "act_time": 832.721,
                                "est_card": 43015,
                                "est_width": 23,
                                "pg_cost": 452845.98,
                                "op_name": "Index Nested Loop",
                                "output_columns": [
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 3,
                                                "child_ref": None,
                                                "table_name": "mc",
                                                "col_name": "company_id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    },
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 32,
                                                "child_ref": None,
                                                "table_name": "mk",
                                                "col_name": "keyword_id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    },
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 43,
                                                "child_ref": None,
                                                "table_name": "n",
                                                "col_name": "name",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    },
                                ],
                                "act_children_card": 972766,
                                "est_children_card": 426825,
                                "above_udf_filter": False,
                                "is_udf_filter": False,
                            },
                            "children": [
                                {
                                    "plan_parameters": {
                                        "act_card": 23726,
                                        "act_time": 509.77,
                                        "est_card": 9485,
                                        "est_width": 31,
                                        "pg_cost": 436975.14,
                                        "op_name": "Index Nested Loop",
                                        "output_columns": [
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 51,
                                                        "child_ref": None,
                                                        "table_name": "ci",
                                                        "col_name": "movie_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 29,
                                                        "child_ref": None,
                                                        "table_name": "mc",
                                                        "col_name": "movie_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 3,
                                                        "child_ref": None,
                                                        "table_name": "mc",
                                                        "col_name": "company_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 43,
                                                        "child_ref": None,
                                                        "table_name": "n",
                                                        "col_name": "name",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 57,
                                                        "child_ref": None,
                                                        "table_name": "t",
                                                        "col_name": "id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                        ],
                                        "act_children_card": 23726,
                                        "est_children_card": 9485,
                                        "above_udf_filter": False,
                                        "is_udf_filter": False,
                                    },
                                    "children": [
                                        {
                                            "plan_parameters": {
                                                "act_card": 23726,
                                                "act_time": 451.752,
                                                "est_card": 9485,
                                                "est_width": 27,
                                                "pg_cost": 432723.34,
                                                "op_name": "Index Nested Loop",
                                                "output_columns": [
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 51,
                                                                "child_ref": None,
                                                                "table_name": "ci",
                                                                "col_name": "movie_id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 29,
                                                                "child_ref": None,
                                                                "table_name": "mc",
                                                                "col_name": "movie_id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 3,
                                                                "child_ref": None,
                                                                "table_name": "mc",
                                                                "col_name": "company_id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 43,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "name",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                ],
                                                "act_children_card": 25000,
                                                "est_children_card": 18135,
                                                "above_udf_filter": False,
                                                "is_udf_filter": False,
                                            },
                                            "children": [
                                                {
                                                    "plan_parameters": {
                                                        "act_card": 12500,
                                                        "act_time": 398.07,
                                                        "est_card": 3627,
                                                        "est_width": 19,
                                                        "pg_cost": 430655.44,
                                                        "op_name": "Index Nested Loop",
                                                        "output_columns": [
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 51,
                                                                        "child_ref": None,
                                                                        "table_name": "ci",
                                                                        "col_name": "movie_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 43,
                                                                        "child_ref": None,
                                                                        "table_name": "n",
                                                                        "col_name": "name",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                        ],
                                                        "act_children_card": 13320,
                                                        "est_children_card": 214338,
                                                        "above_udf_filter": False,
                                                        "is_udf_filter": False,
                                                    },
                                                    "children": [
                                                        {
                                                            "plan_parameters": {
                                                                "act_card": 2664,
                                                                "act_time": 385.853,
                                                                "est_card": 417,
                                                                "est_width": 19,
                                                                "pg_cost": 107707.64,
                                                                "table_name": "name",
                                                                "table_id": 18,
                                                                "op_name": "Seq Scan",
                                                                "filter": {
                                                                    "table_name": "n",
                                                                    "col_name": "name",
                                                                    "col_id": 43,
                                                                    "operator": "LIKE",
                                                                    "literal": "'X%'",
                                                                    "is_index_cond": False,
                                                                    "filter_complexity": 1,
                                                                },
                                                                "output_columns": [
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 42,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 43,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "name",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 44,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "imdb_index",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 45,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "imdb_id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 46,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "gender",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 13,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "name_pcode_cf",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 18,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "name_pcode_nf",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 30,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "surname_pcode",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 56,
                                                                                "child_ref": None,
                                                                                "table_name": "n",
                                                                                "col_name": "md5sum",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                ],
                                                                "act_children_card": 1,
                                                                "est_children_card": 1,
                                                                "above_udf_filter": False,
                                                                "is_udf_filter": False,
                                                            },
                                                            "children": [],
                                                        },
                                                        {
                                                            "plan_parameters": {
                                                                "act_card": 5,
                                                                "act_time": 0.003,
                                                                "est_card": 514,
                                                                "est_width": 8,
                                                                "pg_cost": 769.32,
                                                                "table_name": "cast_info",
                                                                "table_id": 2,
                                                                "op_name": "Index Scan",
                                                                "join": {
                                                                    "table_name1": "ci",
                                                                    "column_name1": "person_id",
                                                                    "col_id1": 50,
                                                                    "operator": "=",
                                                                    "table_name2": "n",
                                                                    "column_name2": "id",
                                                                    "col_id2": 42,
                                                                    "is_index_cond": True,
                                                                },
                                                                "output_columns": [
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 76,
                                                                                "child_ref": None,
                                                                                "table_name": "ci",
                                                                                "col_name": "id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 50,
                                                                                "child_ref": None,
                                                                                "table_name": "ci",
                                                                                "col_name": "person_id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 51,
                                                                                "child_ref": None,
                                                                                "table_name": "ci",
                                                                                "col_name": "movie_id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 77,
                                                                                "child_ref": None,
                                                                                "table_name": "ci",
                                                                                "col_name": "person_role_id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 78,
                                                                                "child_ref": None,
                                                                                "table_name": "ci",
                                                                                "col_name": "note",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 79,
                                                                                "child_ref": None,
                                                                                "table_name": "ci",
                                                                                "col_name": "nr_order",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 52,
                                                                                "child_ref": None,
                                                                                "table_name": "ci",
                                                                                "col_name": "role_id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                ],
                                                                "act_children_card": 1,
                                                                "est_children_card": 1,
                                                                "above_udf_filter": False,
                                                                "is_udf_filter": False,
                                                            },
                                                            "children": [],
                                                        },
                                                    ],
                                                },
                                                {
                                                    "plan_parameters": {
                                                        "act_card": 2,
                                                        "act_time": 0.004,
                                                        "est_card": 5,
                                                        "est_width": 8,
                                                        "pg_cost": 0.52,
                                                        "table_name": "movie_companies",
                                                        "table_id": 14,
                                                        "op_name": "Index Scan",
                                                        "join": {
                                                            "table_name1": "mc",
                                                            "column_name1": "movie_id",
                                                            "col_id1": 29,
                                                            "operator": "=",
                                                            "table_name2": "ci",
                                                            "column_name2": "movie_id",
                                                            "col_id2": 51,
                                                            "is_index_cond": True,
                                                        },
                                                        "output_columns": [
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 28,
                                                                        "child_ref": None,
                                                                        "table_name": "mc",
                                                                        "col_name": "id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 29,
                                                                        "child_ref": None,
                                                                        "table_name": "mc",
                                                                        "col_name": "movie_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 3,
                                                                        "child_ref": None,
                                                                        "table_name": "mc",
                                                                        "col_name": "company_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 16,
                                                                        "child_ref": None,
                                                                        "table_name": "mc",
                                                                        "col_name": "company_type_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 4,
                                                                        "child_ref": None,
                                                                        "table_name": "mc",
                                                                        "col_name": "note",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                        ],
                                                        "act_children_card": 1,
                                                        "est_children_card": 1,
                                                        "above_udf_filter": False,
                                                        "is_udf_filter": False,
                                                    },
                                                    "children": [],
                                                },
                                            ],
                                        },
                                        {
                                            "plan_parameters": {
                                                "act_card": 1,
                                                "act_time": 0.002,
                                                "est_card": 1,
                                                "est_width": 4,
                                                "pg_cost": 0.45,
                                                "table_name": "title",
                                                "table_id": 22,
                                                "op_name": "Index Only Scan",
                                                "join": {
                                                    "table_name1": "t",
                                                    "column_name1": "id",
                                                    "col_id1": 57,
                                                    "operator": "=",
                                                    "table_name2": "ci",
                                                    "column_name2": "movie_id",
                                                    "col_id2": 51,
                                                    "is_index_cond": True,
                                                },
                                                "output_columns": [
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 57,
                                                                "child_ref": None,
                                                                "table_name": "t",
                                                                "col_name": "id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    }
                                                ],
                                                "act_children_card": 1,
                                                "est_children_card": 1,
                                                "above_udf_filter": False,
                                                "is_udf_filter": False,
                                            },
                                            "children": [],
                                        },
                                    ],
                                },
                                {
                                    "plan_parameters": {
                                        "act_card": 41,
                                        "act_time": 0.008,
                                        "est_card": 45,
                                        "est_width": 8,
                                        "pg_cost": 1.22,
                                        "table_name": "movie_keyword",
                                        "table_id": 16,
                                        "op_name": "Index Scan",
                                        "join": {
                                            "table_name1": "mk",
                                            "column_name1": "movie_id",
                                            "col_id1": 8,
                                            "operator": "=",
                                            "table_name2": "ci",
                                            "column_name2": "movie_id",
                                            "col_id2": 51,
                                            "is_index_cond": True,
                                        },
                                        "output_columns": [
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 17,
                                                        "child_ref": None,
                                                        "table_name": "mk",
                                                        "col_name": "id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 8,
                                                        "child_ref": None,
                                                        "table_name": "mk",
                                                        "col_name": "movie_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 32,
                                                        "child_ref": None,
                                                        "table_name": "mk",
                                                        "col_name": "keyword_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                        ],
                                        "act_children_card": 1,
                                        "est_children_card": 1,
                                        "above_udf_filter": False,
                                        "is_udf_filter": False,
                                    },
                                    "children": [],
                                },
                            ],
                        },
                        {
                            "plan_parameters": {
                                "act_card": 234997,
                                "act_time": 62.78,
                                "est_card": 234997,
                                "est_width": 4,
                                "pg_cost": 5344.97,
                                "op_name": "Hash",
                                "output_columns": [
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 14,
                                                "child_ref": None,
                                                "table_name": "cn",
                                                "col_name": "id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    }
                                ],
                                "act_children_card": 234997,
                                "est_children_card": 234997,
                                "above_udf_filter": False,
                                "is_udf_filter": False,
                            },
                            "children": [
                                {
                                    "plan_parameters": {
                                        "act_card": 234997,
                                        "act_time": 28.116,
                                        "est_card": 234997,
                                        "est_width": 4,
                                        "pg_cost": 5344.97,
                                        "table_name": "company_name",
                                        "table_id": 5,
                                        "op_name": "Seq Scan",
                                        "output_columns": [
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 14,
                                                        "child_ref": None,
                                                        "table_name": "cn",
                                                        "col_name": "id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            }
                                        ],
                                        "act_children_card": 1,
                                        "est_children_card": 1,
                                        "above_udf_filter": False,
                                        "is_udf_filter": False,
                                    },
                                    "children": [],
                                }
                            ],
                        },
                    ],
                },
                {
                    "plan_parameters": {
                        "act_card": 1,
                        "act_time": 19.903,
                        "est_card": 1,
                        "est_width": 4,
                        "pg_cost": 8.44,
                        "op_name": "Hash",
                        "output_columns": [
                            {
                                "aggregation": "None",
                                "columns": [
                                    {
                                        "col_id": 26,
                                        "child_ref": None,
                                        "table_name": "k",
                                        "col_name": "id",
                                    }
                                ],
                                "udf_name": None,
                                "udf_output": "False",
                                "child_ref": False,
                            }
                        ],
                        "act_children_card": 1,
                        "est_children_card": 1,
                        "above_udf_filter": False,
                        "is_udf_filter": False,
                    },
                    "children": [
                        {
                            "plan_parameters": {
                                "act_card": 1,
                                "act_time": 19.886,
                                "est_card": 1,
                                "est_width": 4,
                                "pg_cost": 8.44,
                                "table_name": "keyword",
                                "table_id": 9,
                                "op_name": "Index Scan",
                                "filter": {
                                    "table_name": "k",
                                    "col_name": "keyword",
                                    "col_id": 27,
                                    "operator": "=",
                                    "literal": "'character-name-in-title'",
                                    "is_index_cond": True,
                                    "filter_complexity": 0,
                                },
                                "output_columns": [
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 26,
                                                "child_ref": None,
                                                "table_name": "k",
                                                "col_name": "id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    }
                                ],
                                "act_children_card": 1,
                                "est_children_card": 1,
                                "above_udf_filter": False,
                                "is_udf_filter": False,
                            },
                            "children": [],
                        }
                    ],
                },
            ],
        }
    ],
    "sql": "SELECT MIN(n.name) AS member_in_charnamed_movie, MIN(n.name) AS a1 FROM cast_info AS ci, company_name AS cn, keyword AS k, movie_companies AS mc, movie_keyword AS mk, name AS n, title AS t WHERE k.keyword ='character-name-in-title' AND n.name LIKE 'X%' AND n.id = ci.person_id AND ci.movie_id = t.id AND t.id = mk.movie_id AND mk.keyword_id = k.id AND t.id = mc.movie_id AND mc.company_id = cn.id AND ci.movie_id = mc.movie_id AND ci.movie_id = mk.movie_id AND mc.movie_id = mk.movie_id;",
    "dataset": "imdb",
    "plan_runtime_ms": 1182.969,
    "num_tables": 7,
    "num_filters": 2,
    "hint": "/*+Leading(n ci mc t mk cn k)*/",
    "invalid_hint": False,
    "zero_card": False,
    "timeout": False,
    "timeout_threshold": 30000,
}

In [ ]:
plan2 = {
    "plan_parameters": {
        "act_card": 1,
        "act_time": 6773.117,
        "est_card": 1,
        "est_width": 64,
        "pg_cost": 1872.78,
        "op_name": "Simple Aggregate",
        "output_columns": [
            {
                "aggregation": "MIN",
                "columns": [
                    {
                        "col_id": 43,
                        "child_ref": None,
                        "table_name": "n",
                        "col_name": "name",
                    }
                ],
                "udf_name": None,
                "udf_output": "False",
                "child_ref": False,
            },
            {
                "aggregation": "MIN",
                "columns": [
                    {
                        "col_id": 43,
                        "child_ref": None,
                        "table_name": "n",
                        "col_name": "name",
                    }
                ],
                "udf_name": None,
                "udf_output": "False",
                "child_ref": False,
            },
        ],
        "act_children_card": 1918,
        "est_children_card": 1,
        "above_udf_filter": False,
        "is_udf_filter": False,
    },
    "children": [
        {
            "plan_parameters": {
                "act_card": 1918,
                "act_time": 6771.878,
                "est_card": 1,
                "est_width": 15,
                "pg_cost": 1872.76,
                "op_name": "Index Nested Loop",
                "output_columns": [
                    {
                        "aggregation": "None",
                        "columns": [
                            {
                                "col_id": 43,
                                "child_ref": None,
                                "table_name": "n",
                                "col_name": "name",
                            }
                        ],
                        "udf_name": None,
                        "udf_output": "False",
                        "child_ref": False,
                    }
                ],
                "act_children_card": 1918,
                "est_children_card": 1,
                "above_udf_filter": False,
                "is_udf_filter": False,
            },
            "children": [
                {
                    "plan_parameters": {
                        "act_card": 1918,
                        "act_time": 6765.777,
                        "est_card": 1,
                        "est_width": 27,
                        "pg_cost": 1872.32,
                        "op_name": "Index Nested Loop",
                        "output_columns": [
                            {
                                "aggregation": "None",
                                "columns": [
                                    {
                                        "col_id": 51,
                                        "child_ref": None,
                                        "table_name": "ci",
                                        "col_name": "movie_id",
                                    }
                                ],
                                "udf_name": None,
                                "udf_output": "False",
                                "child_ref": False,
                            },
                            {
                                "aggregation": "None",
                                "columns": [
                                    {
                                        "col_id": 29,
                                        "child_ref": None,
                                        "table_name": "mc",
                                        "col_name": "movie_id",
                                    }
                                ],
                                "udf_name": None,
                                "udf_output": "False",
                                "child_ref": False,
                            },
                            {
                                "aggregation": "None",
                                "columns": [
                                    {
                                        "col_id": 8,
                                        "child_ref": None,
                                        "table_name": "mk",
                                        "col_name": "movie_id",
                                    }
                                ],
                                "udf_name": None,
                                "udf_output": "False",
                                "child_ref": False,
                            },
                            {
                                "aggregation": "None",
                                "columns": [
                                    {
                                        "col_id": 43,
                                        "child_ref": None,
                                        "table_name": "n",
                                        "col_name": "name",
                                    }
                                ],
                                "udf_name": None,
                                "udf_output": "False",
                                "child_ref": False,
                            },
                        ],
                        "act_children_card": 1918,
                        "est_children_card": 1,
                        "above_udf_filter": False,
                        "is_udf_filter": False,
                    },
                    "children": [
                        {
                            "plan_parameters": {
                                "act_card": 1918,
                                "act_time": 6758.279,
                                "est_card": 1,
                                "est_width": 31,
                                "pg_cost": 1871.88,
                                "op_name": "Index Nested Loop",
                                "output_columns": [
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 51,
                                                "child_ref": None,
                                                "table_name": "ci",
                                                "col_name": "movie_id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    },
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 29,
                                                "child_ref": None,
                                                "table_name": "mc",
                                                "col_name": "movie_id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    },
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 3,
                                                "child_ref": None,
                                                "table_name": "mc",
                                                "col_name": "company_id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    },
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 8,
                                                "child_ref": None,
                                                "table_name": "mk",
                                                "col_name": "movie_id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    },
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 43,
                                                "child_ref": None,
                                                "table_name": "n",
                                                "col_name": "name",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    },
                                ],
                                "act_children_card": 2000,
                                "est_children_card": 5,
                                "above_udf_filter": False,
                                "is_udf_filter": False,
                            },
                            "children": [
                                {
                                    "plan_parameters": {
                                        "act_card": 250,
                                        "act_time": 6752.893,
                                        "est_card": 1,
                                        "est_width": 23,
                                        "pg_cost": 1871.31,
                                        "op_name": "Index Nested Loop",
                                        "output_columns": [
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 51,
                                                        "child_ref": None,
                                                        "table_name": "ci",
                                                        "col_name": "movie_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 8,
                                                        "child_ref": None,
                                                        "table_name": "mk",
                                                        "col_name": "movie_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 43,
                                                        "child_ref": None,
                                                        "table_name": "n",
                                                        "col_name": "name",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                        ],
                                        "act_children_card": 0,
                                        "est_children_card": 1284,
                                        "above_udf_filter": False,
                                        "is_udf_filter": False,
                                    },
                                    "children": [
                                        {
                                            "plan_parameters": {
                                                "act_card": 1038393,
                                                "act_time": 1894.975,
                                                "est_card": 1284,
                                                "est_width": 12,
                                                "pg_cost": 1284.01,
                                                "op_name": "Index Nested Loop",
                                                "output_columns": [
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 50,
                                                                "child_ref": None,
                                                                "table_name": "ci",
                                                                "col_name": "person_id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 51,
                                                                "child_ref": None,
                                                                "table_name": "ci",
                                                                "col_name": "movie_id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 8,
                                                                "child_ref": None,
                                                                "table_name": "mk",
                                                                "col_name": "movie_id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                ],
                                                "act_children_card": 1046000,
                                                "est_children_card": 1292,
                                                "above_udf_filter": False,
                                                "is_udf_filter": False,
                                            },
                                            "children": [
                                                {
                                                    "plan_parameters": {
                                                        "act_card": 41840,
                                                        "act_time": 58.372,
                                                        "est_card": 34,
                                                        "est_width": 4,
                                                        "pg_cost": 1224.82,
                                                        "op_name": "Index Nested Loop",
                                                        "output_columns": [
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 8,
                                                                        "child_ref": None,
                                                                        "table_name": "mk",
                                                                        "col_name": "movie_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            }
                                                        ],
                                                        "act_children_card": 41840,
                                                        "est_children_card": 302,
                                                        "above_udf_filter": False,
                                                        "is_udf_filter": False,
                                                    },
                                                    "children": [
                                                        {
                                                            "plan_parameters": {
                                                                "act_card": 1,
                                                                "act_time": 0.056,
                                                                "est_card": 1,
                                                                "est_width": 4,
                                                                "pg_cost": 8.44,
                                                                "table_name": "keyword",
                                                                "table_id": 9,
                                                                "op_name": "Index Scan",
                                                                "filter": {
                                                                    "table_name": "k",
                                                                    "col_name": "keyword",
                                                                    "col_id": 27,
                                                                    "operator": "=",
                                                                    "literal": "'character-name-in-title'",
                                                                    "is_index_cond": True,
                                                                    "filter_complexity": 0,
                                                                },
                                                                "output_columns": [
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 26,
                                                                                "child_ref": None,
                                                                                "table_name": "k",
                                                                                "col_name": "id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 27,
                                                                                "child_ref": None,
                                                                                "table_name": "k",
                                                                                "col_name": "keyword",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 97,
                                                                                "child_ref": None,
                                                                                "table_name": "k",
                                                                                "col_name": "phonetic_code",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                ],
                                                                "act_children_card": 1,
                                                                "est_children_card": 1,
                                                                "above_udf_filter": False,
                                                                "is_udf_filter": False,
                                                            },
                                                            "children": [],
                                                        },
                                                        {
                                                            "plan_parameters": {
                                                                "act_card": 41840,
                                                                "act_time": 49.309,
                                                                "est_card": 302,
                                                                "est_width": 8,
                                                                "pg_cost": 1213.36,
                                                                "table_name": "movie_keyword",
                                                                "table_id": 16,
                                                                "op_name": "Index Scan",
                                                                "join": {
                                                                    "table_name1": "mk",
                                                                    "column_name1": "keyword_id",
                                                                    "col_id1": 32,
                                                                    "operator": "=",
                                                                    "table_name2": "k",
                                                                    "column_name2": "id",
                                                                    "col_id2": 26,
                                                                    "is_index_cond": True,
                                                                },
                                                                "output_columns": [
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 17,
                                                                                "child_ref": None,
                                                                                "table_name": "mk",
                                                                                "col_name": "id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 8,
                                                                                "child_ref": None,
                                                                                "table_name": "mk",
                                                                                "col_name": "movie_id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                    {
                                                                        "aggregation": "None",
                                                                        "columns": [
                                                                            {
                                                                                "col_id": 32,
                                                                                "child_ref": None,
                                                                                "table_name": "mk",
                                                                                "col_name": "keyword_id",
                                                                            }
                                                                        ],
                                                                        "udf_name": None,
                                                                        "udf_output": "False",
                                                                        "child_ref": False,
                                                                    },
                                                                ],
                                                                "act_children_card": 1,
                                                                "est_children_card": 1,
                                                                "above_udf_filter": False,
                                                                "is_udf_filter": False,
                                                            },
                                                            "children": [],
                                                        },
                                                    ],
                                                },
                                                {
                                                    "plan_parameters": {
                                                        "act_card": 25,
                                                        "act_time": 0.039,
                                                        "est_card": 38,
                                                        "est_width": 8,
                                                        "pg_cost": 1.36,
                                                        "table_name": "cast_info",
                                                        "table_id": 2,
                                                        "op_name": "Index Scan",
                                                        "join": {
                                                            "table_name1": "ci",
                                                            "column_name1": "movie_id",
                                                            "col_id1": 51,
                                                            "operator": "=",
                                                            "table_name2": "mk",
                                                            "column_name2": "movie_id",
                                                            "col_id2": 8,
                                                            "is_index_cond": True,
                                                        },
                                                        "output_columns": [
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 76,
                                                                        "child_ref": None,
                                                                        "table_name": "ci",
                                                                        "col_name": "id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 50,
                                                                        "child_ref": None,
                                                                        "table_name": "ci",
                                                                        "col_name": "person_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 51,
                                                                        "child_ref": None,
                                                                        "table_name": "ci",
                                                                        "col_name": "movie_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 77,
                                                                        "child_ref": None,
                                                                        "table_name": "ci",
                                                                        "col_name": "person_role_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 78,
                                                                        "child_ref": None,
                                                                        "table_name": "ci",
                                                                        "col_name": "note",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 79,
                                                                        "child_ref": None,
                                                                        "table_name": "ci",
                                                                        "col_name": "nr_order",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                            {
                                                                "aggregation": "None",
                                                                "columns": [
                                                                    {
                                                                        "col_id": 52,
                                                                        "child_ref": None,
                                                                        "table_name": "ci",
                                                                        "col_name": "role_id",
                                                                    }
                                                                ],
                                                                "udf_name": None,
                                                                "udf_output": "False",
                                                                "child_ref": False,
                                                            },
                                                        ],
                                                        "act_children_card": 1,
                                                        "est_children_card": 1,
                                                        "above_udf_filter": False,
                                                        "is_udf_filter": False,
                                                    },
                                                    "children": [],
                                                },
                                            ],
                                        },
                                        {
                                            "plan_parameters": {
                                                "act_card": 0,
                                                "act_time": 0.004,
                                                "est_card": 1,
                                                "est_width": 19,
                                                "pg_cost": 0.46,
                                                "table_name": "name",
                                                "table_id": 18,
                                                "op_name": "Index Scan",
                                                "filter": {
                                                    "table_name": "n",
                                                    "col_name": "name",
                                                    "col_id": 43,
                                                    "operator": "LIKE",
                                                    "literal": "'X%'",
                                                    "is_index_cond": False,
                                                    "filter_complexity": 1,
                                                },
                                                "join": {
                                                    "table_name1": "n",
                                                    "column_name1": "id",
                                                    "col_id1": 42,
                                                    "operator": "=",
                                                    "table_name2": "ci",
                                                    "column_name2": "person_id",
                                                    "col_id2": 50,
                                                    "is_index_cond": True,
                                                },
                                                "output_columns": [
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 42,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 43,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "name",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 44,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "imdb_index",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 45,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "imdb_id",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 46,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "gender",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 13,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "name_pcode_cf",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 18,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "name_pcode_nf",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 30,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "surname_pcode",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                    {
                                                        "aggregation": "None",
                                                        "columns": [
                                                            {
                                                                "col_id": 56,
                                                                "child_ref": None,
                                                                "table_name": "n",
                                                                "col_name": "md5sum",
                                                            }
                                                        ],
                                                        "udf_name": None,
                                                        "udf_output": "False",
                                                        "child_ref": False,
                                                    },
                                                ],
                                                "act_children_card": 1,
                                                "est_children_card": 1,
                                                "above_udf_filter": False,
                                                "is_udf_filter": False,
                                            },
                                            "children": [],
                                        },
                                    ],
                                },
                                {
                                    "plan_parameters": {
                                        "act_card": 8,
                                        "act_time": 0.018,
                                        "est_card": 5,
                                        "est_width": 8,
                                        "pg_cost": 0.52,
                                        "table_name": "movie_companies",
                                        "table_id": 14,
                                        "op_name": "Index Scan",
                                        "join": {
                                            "table_name1": "mc",
                                            "column_name1": "movie_id",
                                            "col_id1": 29,
                                            "operator": "=",
                                            "table_name2": "ci",
                                            "column_name2": "movie_id",
                                            "col_id2": 51,
                                            "is_index_cond": True,
                                        },
                                        "output_columns": [
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 28,
                                                        "child_ref": None,
                                                        "table_name": "mc",
                                                        "col_name": "id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 29,
                                                        "child_ref": None,
                                                        "table_name": "mc",
                                                        "col_name": "movie_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 3,
                                                        "child_ref": None,
                                                        "table_name": "mc",
                                                        "col_name": "company_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 16,
                                                        "child_ref": None,
                                                        "table_name": "mc",
                                                        "col_name": "company_type_id",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                            {
                                                "aggregation": "None",
                                                "columns": [
                                                    {
                                                        "col_id": 4,
                                                        "child_ref": None,
                                                        "table_name": "mc",
                                                        "col_name": "note",
                                                    }
                                                ],
                                                "udf_name": None,
                                                "udf_output": "False",
                                                "child_ref": False,
                                            },
                                        ],
                                        "act_children_card": 1,
                                        "est_children_card": 1,
                                        "above_udf_filter": False,
                                        "is_udf_filter": False,
                                    },
                                    "children": [],
                                },
                            ],
                        },
                        {
                            "plan_parameters": {
                                "act_card": 1,
                                "act_time": 0.003,
                                "est_card": 1,
                                "est_width": 4,
                                "pg_cost": 0.44,
                                "table_name": "company_name",
                                "table_id": 5,
                                "op_name": "Index Only Scan",
                                "join": {
                                    "table_name1": "cn",
                                    "column_name1": "id",
                                    "col_id1": 14,
                                    "operator": "=",
                                    "table_name2": "mc",
                                    "column_name2": "company_id",
                                    "col_id2": 3,
                                    "is_index_cond": True,
                                },
                                "output_columns": [
                                    {
                                        "aggregation": "None",
                                        "columns": [
                                            {
                                                "col_id": 14,
                                                "child_ref": None,
                                                "table_name": "cn",
                                                "col_name": "id",
                                            }
                                        ],
                                        "udf_name": None,
                                        "udf_output": "False",
                                        "child_ref": False,
                                    }
                                ],
                                "act_children_card": 1,
                                "est_children_card": 1,
                                "above_udf_filter": False,
                                "is_udf_filter": False,
                            },
                            "children": [],
                        },
                    ],
                },
                {
                    "plan_parameters": {
                        "act_card": 1,
                        "act_time": 0.003,
                        "est_card": 1,
                        "est_width": 4,
                        "pg_cost": 0.45,
                        "table_name": "title",
                        "table_id": 22,
                        "op_name": "Index Only Scan",
                        "join": {
                            "table_name1": "t",
                            "column_name1": "id",
                            "col_id1": 57,
                            "operator": "=",
                            "table_name2": "ci",
                            "column_name2": "movie_id",
                            "col_id2": 51,
                            "is_index_cond": True,
                        },
                        "output_columns": [
                            {
                                "aggregation": "None",
                                "columns": [
                                    {
                                        "col_id": 57,
                                        "child_ref": None,
                                        "table_name": "t",
                                        "col_name": "id",
                                    }
                                ],
                                "udf_name": None,
                                "udf_output": "False",
                                "child_ref": False,
                            }
                        ],
                        "act_children_card": 1,
                        "est_children_card": 1,
                        "above_udf_filter": False,
                        "is_udf_filter": False,
                    },
                    "children": [],
                },
            ],
        }
    ],
    "sql": "SELECT MIN(n.name) AS member_in_charnamed_movie, MIN(n.name) AS a1 FROM cast_info AS ci, company_name AS cn, keyword AS k, movie_companies AS mc, movie_keyword AS mk, name AS n, title AS t WHERE k.keyword ='character-name-in-title' AND n.name LIKE 'X%' AND n.id = ci.person_id AND ci.movie_id = t.id AND t.id = mk.movie_id AND mk.keyword_id = k.id AND t.id = mc.movie_id AND mc.company_id = cn.id AND ci.movie_id = mc.movie_id AND ci.movie_id = mk.movie_id AND mc.movie_id = mk.movie_id;",
    "dataset": "imdb",
    "plan_runtime_ms": 6773.281,
    "num_tables": 7,
    "num_filters": 2,
    "hint": "",
    "invalid_hint": False,
    "zero_card": False,
    "timeout": False,
    "timeout_threshold": None,
}

In [ ]:
plan3_nocol = {
    "plan_parameters": {
        "act_card": 1,
        "act_time": 6773.117,
        "est_card": 1,
        "est_width": 64,
        "pg_cost": 1872.78,
        "op_name": "Simple Aggregate",
        "output_columns": [],
        "act_children_card": 1918,
        "est_children_card": 1,
        "above_udf_filter": False,
        "is_udf_filter": False,
    },
    "children": [
        {
            "plan_parameters": {
                "act_card": 41840,
                "act_time": 49.309,
                "est_card": 302,
                "est_width": 8,
                "pg_cost": 1213.36,
                "table_name": "movie_keyword",
                "table_id": 16,
                "op_name": "Index Scan",
                "output_columns": [],
                "act_children_card": 1,
                "est_children_card": 1,
                "above_udf_filter": False,
                "is_udf_filter": False,
            },
            "children": [],
        },
    ],
    "sql": "SELECT MIN(n.name) AS member_in_charnamed_movie, MIN(n.name) AS a1 FROM cast_info AS ci, company_name AS cn, keyword AS k, movie_companies AS mc, movie_keyword AS mk, name AS n, title AS t WHERE k.keyword ='character-name-in-title' AND n.name LIKE 'X%' AND n.id = ci.person_id AND ci.movie_id = t.id AND t.id = mk.movie_id AND mk.keyword_id = k.id AND t.id = mc.movie_id AND mc.company_id = cn.id AND ci.movie_id = mc.movie_id AND ci.movie_id = mk.movie_id AND mc.movie_id = mk.movie_id;",
    "dataset": "imdb",
    "plan_runtime_ms": 10000,
    "num_tables": 7,
    "num_filters": 2,
    "hint": "",
    "invalid_hint": False,
    "zero_card": False,
    "timeout": False,
    "timeout_threshold": None,
}

In [ ]:
import functools
from typing import Dict

from classes import DataLoaderOptions
from models.zeroshot.postgres_plan_batching import postgres_plan_collator
from torch.utils.data import DataLoader
from training.batch_to_funcs import batch_to
from training.dataset.dataset_creation import create_datasets

assert feature_stats is not None
assert database_statistics is not None
import pandas as pd


def create_dataloader(
    json_plans: List[Dict],
    model_config: ModelConfig,
    feature_statistics: Dict,
    database_statistics: Dict,
):
    for p in json_plans:
        p["run_file_id"] = 0

    # create dataloader with this plan
    _, test_dataset, _, test_database_statistics = create_datasets(
        None,
        model_config=model_config,
        shuffle_before_split=False,
        val_ratio=0.0,
        is_test_dataset=True,
        inject_plans=json_plans,
        inject_database_statistics=database_statistics,
    )
    # test dataset
    test_collate_fn = functools.partial(
        postgres_plan_collator,
        db_statistics=test_database_statistics,
        feature_statistics=feature_statistics,
        plan_featurization=model_config.featurization,
        test_mode=True,
        skip_top_aggregation=model_config.skip_top_aggregation,
    )

    # previously shuffle=False but this resulted in bugs
    data_loader_options = DataLoaderOptions()
    dataloader_args = dict(
        batch_size=model_config.batch_size,
        shuffle=data_loader_options.shuffle,
        num_workers=1,
        pin_memory=data_loader_options.pin_memory,
    )
    dataloader_args.update(collate_fn=test_collate_fn)
    data_loader = DataLoader(test_dataset, **dataloader_args)

    return data_loader


def inference(plans: List[Dict]):
    dataloader = create_dataloader(
        json_plans=plans,
        model_config=config,
        feature_statistics=feature_stats,
        database_statistics=database_statistics,
    )

    for batch in dataloader:
        (
            input_model,
            label,
            sample_idxs_batch,
            sql,
            hints,
            selectivity,
            scan_op,
            join_op,
        ) = batch_to(batch, "cpu", model.label_norm)

        output = model(input_model)

        if model.label_norm is not None:
            pred = model.label_norm.inverse_transform(output)
            label = model.label_norm.inverse_transform(label.reshape(-1, 1))
        else:
            pred = output

        label = label.reshape(-1).detach().numpy()
        pred = pred.reshape(-1).detach().numpy()

        # create dataframe with col preds, labels
        df = pd.DataFrame({"pred": pred, "label": label})
        print(df)

In [ ]:
def remove_aggregation(plan: Dict):
    if "Aggregate" in plan["plan_parameters"]["op_name"]:
        # copy over relevant infos
        assert len(plan["children"]) == 1
        child = plan["children"][0]

        copy_over_args = [
            "sql",
            "dataset",
            "plan_runtime_ms",
            "num_tables",
            "num_filters",
            "hint",
            "invalid_hint",
            "zero_card",
            "timeout",
            "timeout_threshold",
            "run_file_id",
        ]
        for arg_name in copy_over_args:
            if arg_name not in plan:
                print(f"Arg not found: {arg_name}")
                continue
            child[arg_name] = plan[arg_name]

        return child

    return plan


plan1_no_aggregation = remove_aggregation(plan1)
plan2_no_aggregation = remove_aggregation(plan2)

# inference([plan1, plan2, plan1_no_aggregation, plan2_no_aggregation, plan3_nocol])
inference([plan3_nocol])